# 04 — TF-IDF dan Cosine Similarity

**Project:** SmartLibrary AI  
**Status:** Kerangka notebook, belum diproses.

Membangun mesin rekomendasi berbasis teks tanpa model generatif.


# 04 — Implementasi TF-IDF dan Cosine Similarity

## Tujuan
Membangun sistem rekomendasi buku berdasarkan kemiripan
konten menggunakan Natural Language Processing (NLP).

## Metode
1. Memuat dataset hasil pembersihan.
2. Mengubah teks buku menjadi vektor TF-IDF.
3. Menghitung Cosine Similarity.
4. Menghasilkan maksimal 10 rekomendasi buku.
5. Mengintegrasikan pencarian Indonesia dan Inggris.

## Input
`dataset/processed/books_clean.csv`

## Output
Model TF-IDF dan sistem rekomendasi Top 10.


## Tahapan yang akan dikerjakan

1. Baca dataset bersih dan tetapkan kolom teks rekomendasi.
2. Bangun TfidfVectorizer dengan pengaturan yang dapat diulang.
3. Hitung cosine similarity untuk kueri pengguna.
4. Tampilkan Top 5 buku dan alasan kecocokan kata kunci.
5. Implementasikan rekomendasi buku yang mirip.
6. Simpan vectorizer dan matriks fitur jika sudah tervalidasi.

## Catatan

Tahap ini akan diisi dengan kode per cell dan penjelasan yang lengkap ketika kita memulai notebook terkait. Jangan menyimpan hasil olahan ke `dataset/raw/`.

In [2]:

# =====================================================
# CELL 2 - IMPORT LIBRARY
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("[OK] Seluruh library berhasil diimport.")


[OK] Seluruh library berhasil diimport.


In [3]:

# =====================================================
# CELL 3 - MEMBACA DATASET
# =====================================================

# Mencari folder utama project
current_dir = Path.cwd().resolve()

ROOT = next(
    (
        folder for folder in
        [current_dir, *current_dir.parents]
        if (
            folder / "dataset" / "processed"
            / "books_clean.csv"
        ).is_file()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "File books_clean.csv tidak ditemukan!"
    )

# Lokasi dataset
DATA_PATH = (
    ROOT / "dataset" / "processed" / "books_clean.csv"
)

# Membaca dataset
df_books = pd.read_csv(
    DATA_PATH,
    dtype={
        "isbn13": "string",
        "isbn10": "string"
    }
)

# Validasi kolom penting
required_columns = [
    "isbn13",
    "title",
    "authors",
    "categories",
    "description",
    "combined_text"
]

assert all(
    col in df_books.columns
    for col in required_columns
), "Ada kolom penting yang tidak ditemukan!"

assert df_books["combined_text"].notna().all()
assert (
    df_books["combined_text"].str.strip() != ""
).all()

# Reset index agar sesuai dengan indeks matriks
df_books = df_books.reset_index(drop=True)

print("=" * 50)
print("DATASET SMARTLIBRARY AI")
print("=" * 50)

print("Jumlah buku  :", len(df_books))
print("Jumlah kolom :", len(df_books.columns))

print("\n[OK] Dataset siap digunakan.")

display(
    df_books[
        ["title", "authors", "categories", "combined_text"]
    ].head()
)


DATASET SMARTLIBRARY AI
Jumlah buku  : 5094
Jumlah kolom : 24

[OK] Dataset siap digunakan.


,title,authors,categories,combined_text
0,10 Lb. Penalty,Dick Francis,Fiction,10 lb penalty fiction entering the political a...
1,1000 Rings,Marthe Le Van,Crafts & Hobbies,1000 rings crafts hobbies an extensive and div...
2,1001 Illuminated Initial Letters,Owen Jones,Design,1001 illuminated initial letters design the il...
3,1421,Gavin Menzies,History,1421 history based on 15 years of in depth res...
4,1491,Charles C. Mann,History,1491 history describes how recent archaeologic...


In [4]:

# =====================================================
# CELL 4 - PEMBENTUKAN MATRIKS TF-IDF
# =====================================================

# Konfigurasi TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    max_features=40000,
    sublinear_tf=True,
    strip_accents="unicode"
)

# Mengambil teks yang telah dibersihkan
book_texts = df_books["combined_text"].astype(str)

# Mempelajari kosakata dan membentuk matriks
tfidf_matrix = tfidf_vectorizer.fit_transform(
    book_texts
)

# Informasi hasil
print("=" * 50)
print("HASIL PEMBENTUKAN TF-IDF")
print("=" * 50)

print("Jumlah buku       :", tfidf_matrix.shape[0])
print("Jumlah fitur      :", tfidf_matrix.shape[1])
print("Bentuk matriks    :", tfidf_matrix.shape)

print(
    "Jumlah nilai nonzero:",
    tfidf_matrix.nnz
)

print("\n[OK] Matriks TF-IDF berhasil dibuat.")


HASIL PEMBENTUKAN TF-IDF
Jumlah buku       : 5094
Jumlah fitur      : 40000
Bentuk matriks    : (5094, 40000)
Jumlah nilai nonzero: 253425

[OK] Matriks TF-IDF berhasil dibuat.


In [5]:

# =====================================================
# CELL 5 - VEKTORISASI QUERY PENGGUNA
# =====================================================

# Contoh permintaan buku dalam bahasa Inggris
user_query = (
    "I want a mystery novel about "
    "murder and detective investigations"
)

# Normalisasi query sesuai format dataset
import re

normalized_query = re.sub(
    r"[^\w\s]", " ", user_query.casefold()
)
normalized_query = re.sub(
    r"\s+", " ", normalized_query
).strip()

# Mengubah query menjadi vektor TF-IDF
query_vector = tfidf_vectorizer.transform(
    [normalized_query]
)

print("=" * 50)
print("VEKTORISASI QUERY")
print("=" * 50)

print("Input asli      :", user_query)
print("Normalisasi     :", normalized_query)
print("Bentuk vektor   :", query_vector.shape)
print("Fitur aktif     :", query_vector.nnz)

assert query_vector.nnz > 0, (
    "Query tidak memiliki kata yang dikenal TF-IDF!"
)

print("\n[OK] Query berhasil diubah menjadi vektor.")


VEKTORISASI QUERY
Input asli      : I want a mystery novel about murder and detective investigations
Normalisasi     : i want a mystery novel about murder and detective investigations
Bentuk vektor   : (1, 40000)
Fitur aktif     : 6

[OK] Query berhasil diubah menjadi vektor.


In [6]:

# =====================================================
# CELL 6 - PERHITUNGAN COSINE SIMILARITY
# =====================================================

# Menghitung kemiripan query dengan setiap buku
similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()

# Membuat tabel hasil perhitungan
df_similarity = df_books[
    [
        "isbn13",
        "title",
        "authors",
        "categories"
    ]
].copy()

# Menambahkan skor kemiripan
df_similarity["similarity_score"] = similarity_scores

# Mengurutkan berdasarkan skor tertinggi
df_similarity = df_similarity.sort_values(
    by="similarity_score",
    ascending=False
)

# Menampilkan 10 buku dengan skor > 0
top_10 = (
    df_similarity[
        df_similarity["similarity_score"] > 0
    ]
    .head(10)
    .copy()
)

# Menampilkan hasil
print("=" * 55)
print("TOP 10 REKOMENDASI BUKU")
print("=" * 55)

print("Query:", user_query)
print("Jumlah buku dibandingkan:", len(df_books))
print("Jumlah buku dengan skor > 0:",
      (similarity_scores > 0).sum())

if top_10.empty:
    print("\nTidak ditemukan buku dengan kecocokan kata.")
else:
    display(
        top_10.style.format({
            "similarity_score": "{:.4f}"
        })
    )

print("\n[OK] Perhitungan Cosine Similarity selesai.")


TOP 10 REKOMENDASI BUKU
Query: I want a mystery novel about murder and detective investigations
Jumlah buku dibandingkan: 5094
Jumlah buku dengan skor > 0: 869


,isbn13,title,authors,categories,similarity_score
907,9780425198322,Death's Acre,Bill Bass;William M. Bass;Jon Jefferson,True Crime,0.1983
2510,9780631231271,Philosophical Investigations,Ludwig Wittgenstein,Philosophy,0.1921
2115,9780718149291,Medicus and the Disappearing Dancing Girls,Ruth Downie,Detective and mystery stories,0.1724
3961,9780007154883,The Listerdale Mystery,Agatha Christie,Detective and mystery stories,0.1543
4785,9780517119129,Two Complete Novels,Douglas Adams,"Detective and mystery stories, English.",0.1492
3822,9780007121021,The Hollow,Agatha Christie,Detective and mystery stories,0.1437
118,9780007120963,A Murder is Announced,Agatha Christie,"Detective and mystery stories, English",0.1352
4256,9781569471036,The Rattle-Rat,Janwillem Van de Wetering,Fiction,0.1199
3018,9780393329124,Spook: Science Tackles the Afterlife,Mary Roach,Language Arts & Disciplines,0.1146
2233,9780007113804,Murder in Mesopotamia,Agatha Christie,Detective and mystery stories,0.1139



[OK] Perhitungan Cosine Similarity selesai.


In [7]:

# =====================================================
# CELL 7 - FUNGSI REKOMENDASI TOP 10 BUKU
# =====================================================

import sys

# Mengakses modul dari folder src
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bilingual import normalize_query


def recommend_books(query):
    """
    Menghasilkan maksimal 10 rekomendasi buku
    berdasarkan TF-IDF dan Cosine Similarity.

    Parameter
    ---------
    query : str
        Permintaan buku dalam bahasa Inggris.

    Returns
    -------
    DataFrame
        Buku yang diurutkan berdasarkan kemiripan.
    """

    # 1. Normalisasi input
    clean_query = normalize_query(query)

    # Struktur hasil jika pencarian kosong
    result_columns = [
        "isbn13",
        "title",
        "authors",
        "categories",
        "description",
        "thumbnail"
    ]

    empty_result = df_books[
        result_columns
    ].iloc[0:0].copy()

    empty_result["similarity_score"] = pd.Series(
        dtype=float
    )

    # 2. Menangani input kosong
    if not clean_query:
        return empty_result

    # 3. Mengubah query menjadi vektor TF-IDF
    query_vector = tfidf_vectorizer.transform(
        [clean_query]
    )

    # 4. Memeriksa apakah ada fitur yang dikenali
    if query_vector.nnz == 0:
        return empty_result

    # 5. Menghitung Cosine Similarity
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).ravel()

    # 6. Mengambil indeks dengan skor > 0
    valid_indices = np.flatnonzero(scores > 0)

    # 7. Mengurutkan berdasarkan skor tertinggi
    sorted_indices = valid_indices[
        np.argsort(
            -scores[valid_indices],
            kind="stable"
        )
    ]

    # 8. Mengambil maksimal 10 buku
    top_indices = sorted_indices[:10]

    # 9. Mengambil informasi buku
    recommendations = df_books.iloc[
        top_indices
    ][result_columns].copy()

    # 10. Menambahkan skor kemiripan
    recommendations["similarity_score"] = (
        scores[top_indices]
    )

    # 11. Merapikan indeks
    recommendations = recommendations.reset_index(
        drop=True
    )

    return recommendations


print("[OK] Fungsi recommend_books() berhasil dibuat.")


[OK] Fungsi recommend_books() berhasil dibuat.


In [8]:

# =====================================================
# CELL 8 - PENGUJIAN BERBAGAI TOPIK BUKU
# =====================================================

test_queries = [
    "Mystery novels about murder and detectives",
    "Books about psychology and human behavior",
    "Books about world history"
]

for query in test_queries:

    print("=" * 70)
    print("PENCARIAN:", query)
    print("=" * 70)

    results = recommend_books(query)

    print("Jumlah rekomendasi:", len(results))

    if results.empty:
        print("Tidak ditemukan buku yang cocok.")
    else:
        preview = results[
            [
                "title",
                "authors",
                "categories",
                "similarity_score"
            ]
        ].copy()

        preview.index = range(
            1, len(preview) + 1
        )

        display(
            preview.style.format({
                "similarity_score": "{:.4f}"
            })
        )


PENCARIAN: Mystery novels about murder and detectives
Jumlah rekomendasi: 10


,title,authors,categories,similarity_score
1,Seventy-Seven Clocks,Christopher Fowler,Fiction,0.1901
2,Two Complete Novels,Douglas Adams,"Detective and mystery stories, English.",0.1390
3,The Listerdale Mystery,Agatha Christie,Detective and mystery stories,0.1326
4,Cat Laughing Last,Shirley Rousseau Murphy,Fiction,0.1274
5,Murder in Foggy Bottom,Margaret Truman,Fiction,0.1226
6,"Harry Bosch Novels, The: Volume 2",Michael Connelly,Fiction,0.1137
7,The Sigma Protocol,Robert Ludlum,Fiction,0.1108
8,A Murder is Announced,Agatha Christie,"Detective and mystery stories, English",0.1053
9,Death's Acre,Bill Bass;William M. Bass;Jon Jefferson,True Crime,0.1049
10,Mystery Play,Grant Morrison;Jon J. Muth,Comics & Graphic Novels,0.1027


PENCARIAN: Books about psychology and human behavior
Jumlah rekomendasi: 10


,title,authors,categories,similarity_score
1,Myth and Reality,Mircea Eliade,Religion,0.3099
2,The Nature of Play,Anthony D. Pellegrini;Peter K. Smith,Psychology,0.1918
3,Authentic Happiness,Martin E. P. Seligman,Psychology,0.1863
4,The Crowd: Extradordinary Popular Delusions: Twin Classics of Crowd Pyschology,Gustave Le Bon;Charles Mackay,Psychology,0.1788
5,The Whore's Child and Other Stories,Richard Russo,Human behavior,0.1608
6,DREAM & THE UNDERWOR,James Hillman,Psychology,0.1577
7,Extraordinary Popular Delusions and the Madness of Crowds and ConfusiÃ3n de Confusiones,Charles Mackay,Business & Economics,0.1363
8,The Search for the Beloved,Jean Houston,"Body, Mind & Spirit",0.1216
9,The Mating Mind,Geoffrey Miller,Psychology,0.1180
10,The Denial of Death,Ernest Becker,Philosophy,0.1175


PENCARIAN: Books about world history
Jumlah rekomendasi: 10


,title,authors,categories,similarity_score
1,The Basque History of the World,Mark Kurlansky,History,0.2625
2,The War of the World,Niall Ferguson,History,0.2446
3,The Classical World,Robin Lane Fox,History,0.2372
4,The Island At The Center Of The World,Russell Shorto,History,0.2163
5,The Boomer Bible,R. F. Laird;Robert F. Laird,Humor,0.2148
6,A History of the World in 6 Glasses,Tom Standage,Cooking,0.2026
7,Late Victorian Holocausts: El Nino Famines and the Making of the Third World,Mike Davis,History,0.1713
8,"Since 1500, Chapters 15-30",Richard W. Bulliet,Science,0.1514
9,A Crack in the Edge of the World,Simon Winchester,History,0.1443
10,"Legends, Lies & Cherished Myths of World History",Richard Shenkman,History,0.1317


In [9]:

# =====================================================
# CELL 9 - VALIDASI FUNGSI REKOMENDASI
# =====================================================

# Pengujian input normal
result_normal = recommend_books(
    "Mystery detective novels"
)

# Pengujian input kosong
result_empty = recommend_books("")

# Pengujian input yang tidak ada dalam kosakata
result_unknown = recommend_books(
    "zzzxxyyqqq"
)

# Validasi
assert len(result_normal) <= 10, (
    "Jumlah rekomendasi melebihi 10!"
)

assert len(result_empty) == 0, (
    "Input kosong seharusnya tidak menghasilkan buku!"
)

assert len(result_unknown) == 0, (
    "Query tanpa fitur cocok seharusnya tidak menghasilkan buku!"
)

assert (
    result_normal["similarity_score"] > 0
).all(), "Ditemukan rekomendasi dengan skor 0!"

assert result_normal[
    "similarity_score"
].is_monotonic_decreasing, (
    "Urutan skor rekomendasi tidak sesuai!"
)

print("=" * 55)
print("VALIDASI FUNGSI REKOMENDASI")
print("=" * 55)

print("[OK] Maksimal 10 rekomendasi.")
print("[OK] Input kosong ditangani.")
print("[OK] Query tidak dikenal ditangani.")
print("[OK] Skor rekomendasi lebih dari 0.")
print("[OK] Urutan berdasarkan skor tertinggi.")

print("\n[OK] Seluruh validasi berhasil.")


VALIDASI FUNGSI REKOMENDASI
[OK] Maksimal 10 rekomendasi.
[OK] Input kosong ditangani.
[OK] Query tidak dikenal ditangani.
[OK] Skor rekomendasi lebih dari 0.
[OK] Urutan berdasarkan skor tertinggi.

[OK] Seluruh validasi berhasil.


In [10]:

# =====================================================
# CELL 10 - INTEGRASI REKOMENDASI BILINGUAL
# =====================================================

from src.bilingual import prepare_bilingual_query


def recommend_books_bilingual(query, language="auto"):
    """
    Menghasilkan maksimal 10 rekomendasi buku
    menggunakan input Indonesia atau Inggris.

    Parameters
    ----------
    query : str
        Permintaan buku dari pengguna.

    language : str
        'id'   : Bahasa Indonesia
        'en'   : Bahasa Inggris
        'auto' : Deteksi otomatis

    Returns
    -------
    tuple
        Informasi pemrosesan query dan
        DataFrame hasil rekomendasi.
    """

    # 1. Menyiapkan query bahasa Inggris
    query_info = prepare_bilingual_query(
        query,
        language=language
    )

    # 2. Mengambil teks hasil normalisasi
    processed_query = query_info["normalized_query"]

    # 3. Menggunakan fungsi rekomendasi
    recommendations = recommend_books(
        processed_query
    )

    return query_info, recommendations


print("[OK] Fungsi rekomendasi bilingual berhasil dibuat.")


[OK] Fungsi rekomendasi bilingual berhasil dibuat.


In [11]:

# =====================================================
# CELL 11 - PENGUJIAN REKOMENDASI BAHASA INDONESIA
# =====================================================

user_query = (
    "Saya ingin membaca novel misteri "
    "tentang pembunuhan"
)

# Menggunakan bahasa Indonesia secara eksplisit
query_info, recommendations = (
    recommend_books_bilingual(
        user_query,
        language="id"
    )
)

print("=" * 60)
print("SMARTLIBRARY AI - REKOMENDASI BILINGUAL")
print("=" * 60)

print("Input asli       :", query_info["original_query"])
print("Bahasa           :", query_info["detected_language"])
print("Hasil terjemahan :", query_info["english_query"])
print("Query TF-IDF     :", query_info["normalized_query"])

print("\nJumlah rekomendasi:", len(recommendations))

# Menampilkan rekomendasi
if recommendations.empty:
    print("Tidak ditemukan buku yang cocok.")
else:
    result_view = recommendations[
        [
            "title",
            "authors",
            "categories",
            "similarity_score"
        ]
    ].copy()

    result_view.index = range(
        1, len(result_view) + 1
    )

    display(
        result_view.style.format({
            "similarity_score": "{:.4f}"
        })
    )


SMARTLIBRARY AI - REKOMENDASI BILINGUAL
Input asli       : Saya ingin membaca novel misteri tentang pembunuhan
Bahasa           : id
Hasil terjemahan : I want to read a mystery novel about murder.
Query TF-IDF     : a mystery novel about murder

Jumlah rekomendasi: 10


,title,authors,categories,similarity_score
1,The Listerdale Mystery,Agatha Christie,Detective and mystery stories,0.1983
2,Dance Dance Dance,Haruki Murakami,Crime investigation,0.1885
3,Murder in Foggy Bottom,Margaret Truman,Fiction,0.1833
4,The Sigma Protocol,Robert Ludlum,Fiction,0.1657
5,A Murder is Announced,Agatha Christie,"Detective and mystery stories, English",0.1574
6,Death's Acre,Bill Bass;William M. Bass;Jon Jefferson,True Crime,0.1569
7,Monkeewrench,P. J. Tracy,Fiction,0.1515
8,The Grotesque,Patrick McGrath,Fiction,0.1442
9,The Artist of the Missing,Paul LaFarge,Fiction,0.1434
10,A Kiss Before Dying,Ira Levin,Fiction,0.1425


In [12]:

# =====================================================
# CELL 12 - PERBANDINGAN REKOMENDASI DUA BAHASA
# =====================================================

query_id = (
    "Saya ingin buku tentang psikologi "
    "dan perilaku manusia"
)

query_en = (
    "I want books about psychology "
    "and human behavior"
)

# Rekomendasi bahasa Indonesia
info_id, results_id = recommend_books_bilingual(
    query_id,
    language="id"
)

# Rekomendasi bahasa Inggris
info_en, results_en = recommend_books_bilingual(
    query_en,
    language="en"
)

print("=" * 60)
print("PERBANDINGAN PENCARIAN BILINGUAL")
print("=" * 60)

print("\nBAHASA INDONESIA")
print("Query diproses:", info_id["normalized_query"])

print("\nBAHASA INGGRIS")
print("Query diproses:", info_en["normalized_query"])

# Mengambil daftar ISBN hasil rekomendasi
isbn_id = set(results_id["isbn13"].astype(str))
isbn_en = set(results_en["isbn13"].astype(str))

# Menghitung buku yang muncul dalam kedua hasil
common_books = isbn_id.intersection(isbn_en)

print("\nJumlah rekomendasi Indonesia :", len(results_id))
print("Jumlah rekomendasi Inggris   :", len(results_en))
print("Buku yang muncul pada keduanya:", len(common_books))

# Menampilkan hasil perbandingan
comparison = pd.DataFrame({
    "Peringkat": range(1, 11),
    "Hasil Indonesia": (
        results_id["title"]
        .reindex(range(10))
        .tolist()
    ),
    "Hasil Inggris": (
        results_en["title"]
        .reindex(range(10))
        .tolist()
    )
})

display(comparison)


PERBANDINGAN PENCARIAN BILINGUAL

BAHASA INDONESIA
Query diproses: psychology and human behavior

BAHASA INGGRIS
Query diproses: psychology and human behavior

Jumlah rekomendasi Indonesia : 10
Jumlah rekomendasi Inggris   : 10
Buku yang muncul pada keduanya: 10


,Peringkat,Hasil Indonesia,Hasil Inggris
0,1,Myth and Reality,Myth and Reality
1,2,The Nature of Play,The Nature of Play
2,3,Authentic Happiness,Authentic Happiness
3,4,The Whore's Child and Other Stories,The Whore's Child and Other Stories
4,5,DREAM & THE UNDERWOR,DREAM & THE UNDERWOR
5,6,The Crowd: Extradordinary Popular Delusions: T...,The Crowd: Extradordinary Popular Delusions: T...
6,7,The Mating Mind,The Mating Mind
7,8,The Denial of Death,The Denial of Death
8,9,The Principles of Psychology,The Principles of Psychology
9,10,CliffsNotes On Joyce's Dubliners,CliffsNotes On Joyce's Dubliners


In [13]:

import importlib
import src.bilingual as bilingual

# Memuat ulang file bilingual.py yang telah diperbaiki
importlib.reload(bilingual)

# Memperbarui referensi fungsi
prepare_bilingual_query = bilingual.prepare_bilingual_query

query_id = "Saya ingin buku tentang psikologi dan perilaku manusia"
query_en = "I want books about psychology and human behavior"

info_id = prepare_bilingual_query(
    query_id,
    language="id"
)

info_en = prepare_bilingual_query(
    query_en,
    language="en"
)

print("Hasil Indonesia:", info_id["normalized_query"])
print("Hasil Inggris  :", info_en["normalized_query"])

print(
    "Query identik  :",
    info_id["normalized_query"] == info_en["normalized_query"]
)


Hasil Indonesia: psychology and human behavior
Hasil Inggris  : psychology and human behavior
Query identik  : True


In [14]:

# =====================================================
# CELL 14 - VALIDASI KONSISTENSI REKOMENDASI
# =====================================================

# Menghasilkan rekomendasi dari kedua bahasa
info_id, results_id = recommend_books_bilingual(
    query_id,
    language="id"
)

info_en, results_en = recommend_books_bilingual(
    query_en,
    language="en"
)

# Mengambil identitas buku
isbn_id = results_id["isbn13"].astype(str).tolist()
isbn_en = results_en["isbn13"].astype(str).tolist()

# Menghitung kesamaan rekomendasi
common_books = len(set(isbn_id) & set(isbn_en))

overlap = (
    common_books / 10 * 100
)

# Menampilkan hasil
print("=" * 55)
print("VALIDASI KONSISTENSI BILINGUAL")
print("=" * 55)

print("Jumlah rekomendasi ID :", len(isbn_id))
print("Jumlah rekomendasi EN :", len(isbn_en))
print("Jumlah buku yang sama :", common_books)
print(f"Top-10 Overlap        : {overlap:.2f}%")
print("Urutan identik        :", isbn_id == isbn_en)

# Membandingkan judul buku
comparison = pd.DataFrame({
    "Peringkat": range(1, 11),
    "Bahasa Indonesia": results_id["title"]
        .reindex(range(10)).tolist(),
    "Bahasa Inggris": results_en["title"]
        .reindex(range(10)).tolist()
})

display(comparison)

# Validasi
assert len(isbn_id) == 10
assert len(isbn_en) == 10
assert isbn_id == isbn_en, (
    "Rekomendasi bilingual belum konsisten!"
)

print("\n[OK] Rekomendasi kedua bahasa identik.")


VALIDASI KONSISTENSI BILINGUAL
Jumlah rekomendasi ID : 10
Jumlah rekomendasi EN : 10
Jumlah buku yang sama : 10
Top-10 Overlap        : 100.00%
Urutan identik        : True


,Peringkat,Bahasa Indonesia,Bahasa Inggris
0,1,Myth and Reality,Myth and Reality
1,2,The Nature of Play,The Nature of Play
2,3,Authentic Happiness,Authentic Happiness
3,4,The Whore's Child and Other Stories,The Whore's Child and Other Stories
4,5,DREAM & THE UNDERWOR,DREAM & THE UNDERWOR
5,6,The Crowd: Extradordinary Popular Delusions: T...,The Crowd: Extradordinary Popular Delusions: T...
6,7,The Mating Mind,The Mating Mind
7,8,The Denial of Death,The Denial of Death
8,9,The Principles of Psychology,The Principles of Psychology
9,10,CliffsNotes On Joyce's Dubliners,CliffsNotes On Joyce's Dubliners



[OK] Rekomendasi kedua bahasa identik.


In [15]:

# =====================================================
# CELL 15 - PENYIMPANAN MODEL TF-IDF
# =====================================================

import joblib
from pathlib import Path

# 1. Menentukan folder penyimpanan model
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Menentukan nama file model
MODEL_PATH = MODEL_DIR / "smartlibrary_tfidf.joblib"

# 3. Validasi sebelum menyimpan
assert tfidf_matrix.shape[0] == len(df_books), (
    "Jumlah baris TF-IDF tidak sesuai dataset!"
)

assert df_books["isbn13"].is_unique, (
    "Terdapat ISBN duplikat!"
)

# 4. Menyiapkan komponen model
model_bundle = {
    "vectorizer": tfidf_vectorizer,
    "tfidf_matrix": tfidf_matrix,
    "book_isbns": df_books["isbn13"].astype(str).tolist(),
    "model_name": "SmartLibrary AI - TF-IDF",
    "max_recommendations": 10
}

# 5. Menyimpan model
joblib.dump(
    model_bundle,
    MODEL_PATH,
    compress=3
)

# 6. Menampilkan informasi
print("=" * 55)
print("PENYIMPANAN MODEL SMARTLIBRARY AI")
print("=" * 55)

print("Lokasi model :", MODEL_PATH)
print("Jumlah buku  :", tfidf_matrix.shape[0])
print("Jumlah fitur :", tfidf_matrix.shape[1])

print(
    "Ukuran model :",
    round(MODEL_PATH.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\n[OK] Model TF-IDF berhasil disimpan.")


PENYIMPANAN MODEL SMARTLIBRARY AI
Lokasi model : D:\Project\SmartLibrary_AI\models\smartlibrary_tfidf.joblib
Jumlah buku  : 5094
Jumlah fitur : 40000
Ukuran model : 2.49 MB

[OK] Model TF-IDF berhasil disimpan.


In [16]:

# =====================================================
# CELL 16 - VALIDASI MODEL TERSIMPAN
# =====================================================

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Membaca kembali model
loaded_model = joblib.load(MODEL_PATH)

loaded_vectorizer = loaded_model["vectorizer"]
loaded_matrix = loaded_model["tfidf_matrix"]

# 2. Memastikan ukuran model sesuai
assert loaded_matrix.shape == tfidf_matrix.shape, (
    "Ukuran matriks tidak sesuai!"
)

# 3. Memastikan urutan buku sesuai dengan dataset
saved_isbns = loaded_model["book_isbns"]
current_isbns = df_books["isbn13"].astype(str).tolist()

assert saved_isbns == current_isbns, (
    "Urutan ISBN model berbeda dengan dataset!"
)

# 4. Menguji query menggunakan model tersimpan
test_query = "psychology and human behavior"

query_vector_loaded = loaded_vectorizer.transform(
    [test_query]
)

scores_loaded = cosine_similarity(
    query_vector_loaded,
    loaded_matrix
).ravel()

# 5. Membandingkan dengan model di memori
query_vector_original = tfidf_vectorizer.transform(
    [test_query]
)

scores_original = cosine_similarity(
    query_vector_original,
    tfidf_matrix
).ravel()

assert np.allclose(scores_loaded, scores_original), (
    "Hasil perhitungan model tersimpan tidak sesuai!"
)

# 6. Menampilkan Top 10 dari model tersimpan
top_indices = np.argsort(
    -scores_loaded,
    kind="stable"
)

top_indices = [
    idx for idx in top_indices
    if scores_loaded[idx] > 0
][:10]

results_loaded = df_books.iloc[
    top_indices
][
    ["title", "authors", "categories"]
].copy()

results_loaded["similarity_score"] = (
    scores_loaded[top_indices]
)

results_loaded = results_loaded.reset_index(drop=True)

# 7. Ringkasan validasi
print("=" * 55)
print("VALIDASI MODEL TERSIMPAN")
print("=" * 55)

print("[OK] File model berhasil dibaca.")
print("[OK] Ukuran matriks sesuai.")
print("[OK] Urutan ISBN sesuai.")
print("[OK] Hasil similarity identik.")

print("\nTOP 10 REKOMENDASI DARI MODEL TERSIMPAN")

display(
    results_loaded.style.format({
        "similarity_score": "{:.4f}"
    })
)

print("\n[OK] Model siap digunakan di aplikasi.")


VALIDASI MODEL TERSIMPAN
[OK] File model berhasil dibaca.
[OK] Ukuran matriks sesuai.
[OK] Urutan ISBN sesuai.
[OK] Hasil similarity identik.

TOP 10 REKOMENDASI DARI MODEL TERSIMPAN


,title,authors,categories,similarity_score
0,Myth and Reality,Mircea Eliade,Religion,0.3268
1,The Nature of Play,Anthony D. Pellegrini;Peter K. Smith,Psychology,0.2022
2,Authentic Happiness,Martin E. P. Seligman,Psychology,0.1965
3,The Whore's Child and Other Stories,Richard Russo,Human behavior,0.1696
4,DREAM & THE UNDERWOR,James Hillman,Psychology,0.1662
5,The Crowd: Extradordinary Popular Delusions: Twin Classics of Crowd Pyschology,Gustave Le Bon;Charles Mackay,Psychology,0.1470
6,The Mating Mind,Geoffrey Miller,Psychology,0.1244
7,The Denial of Death,Ernest Becker,Philosophy,0.1238
8,The Principles of Psychology,William James,Psychology,0.1188
9,CliffsNotes On Joyce's Dubliners,Adam Sexton,Literary Criticism,0.1154



[OK] Model siap digunakan di aplikasi.
